# Lab 11: APIM + Managed Identity로 Microsoft Graph 호출

이 노트북은 클라이언트가 **APIM 구독 키만으로** Microsoft Graph를 조회할 수 있음을 검증합니다.
클라이언트는 Graph용 Bearer 토큰을 **직접 발급하지 않습니다** — APIM의 Managed Identity가
내부에서 토큰을 발급해 백엔드로 주입합니다.

- **Part 1 (옵션 A)**: System MI 1개 + 정책 게이트 → 구독별 Operation을 403으로 차단
- **Part 2 (옵션 B)**: 권한별 UAMI 2개 → 잘못된 경로는 토큰 자체에 권한이 없어 Graph가 거부

> 실행 전 `.env`에 `APIM_KEY_GRAPH_USERS`, `APIM_KEY_GRAPH_MAIL`가 입력되어 있어야 합니다.

In [ ]:
# 셀 1: 환경 변수 로드
import os, requests
from pathlib import Path

def load_env(path=".env"):
    env = {}
    p = Path(path)
    if not p.exists():
        p = Path("../../.env")
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env()
APIM_URL = env.get("APIM_URL", "").rstrip("/")
KEY_USERS = env.get("APIM_KEY_GRAPH_USERS", "")
KEY_MAIL = env.get("APIM_KEY_GRAPH_MAIL", "")
TEST_USER_ID = env.get("GRAPH_TEST_USER_ID", "")  # 비우면 셀 3에서 첫 사용자 id 자동 사용

assert APIM_URL, "APIM_URL이 .env에 없습니다."
assert KEY_USERS and "<" not in KEY_USERS, "APIM_KEY_GRAPH_USERS를 .env에 입력하세요."
assert KEY_MAIL and "<" not in KEY_MAIL, "APIM_KEY_GRAPH_MAIL을 .env에 입력하세요."
print("APIM_URL:", APIM_URL)
print("graph-users 키 로드:", KEY_USERS[:4] + "..." )
print("graph-mail  키 로드:", KEY_MAIL[:4] + "...")
GRAPH_BASE = f"{APIM_URL}/graph"
print("Graph API base:", GRAPH_BASE)

## 개념 확인: 클라이언트는 Graph 토큰을 보내지 않는다

아래 모든 요청은 `Ocp-Apim-Subscription-Key` 헤더만 사용합니다.
`Authorization: Bearer ...` 헤더는 **어디에도 없습니다**. Graph 토큰 발급은 APIM MI의 몫입니다.

In [ ]:
# 셀 2: 요청 헬퍼 — 구독 키만 사용 (Bearer 토큰 없음)
def graph_get(path, sub_key, params=None):
    url = f"{GRAPH_BASE}{path}"
    headers = {"Ocp-Apim-Subscription-Key": sub_key}  # Graph Bearer 토큰 없음!
    r = requests.get(url, headers=headers, params=params, timeout=30)
    print(f"GET {path}  → HTTP {r.status_code}")
    assert "authorization" not in {k.lower() for k in headers}, "클라이언트가 Bearer를 보내면 안 됩니다"
    return r

print("헬퍼 준비 완료. 클라이언트 요청 헤더에는 Graph Bearer 토큰이 없습니다.")

## Part 1 (옵션 A): System MI + 정책 게이트

`graph-users` 구독은 `/users`만, `graph-mail` 구독은 `/users/{id}/messages`만 허용됩니다.
격리는 **APIM 정책**이 담당합니다 (실제 토큰 권한은 System MI에 모두 존재).

In [ ]:
# 셀 3: [graph-users 키] GET /users → 200 기대
r = graph_get("/users", KEY_USERS, params={"$top": 3})
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
users = r.json().get("value", [])
print(f"사용자 {len(users)}명 조회 성공")
if not TEST_USER_ID and users:
    TEST_USER_ID = users[0]["id"]
print("TEST_USER_ID:", TEST_USER_ID)

In [ ]:
# 셀 4: [graph-users 키] GET /users/{id}/messages → 403 기대 (정책 차단)
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS)
assert r.status_code == 403, f"기대 403(정책 차단), 실제 {r.status_code}: {r.text[:300]}"
print("✅ 예상대로 차단됨 — graph-users 구독은 메일 조회 불가 (APIM 정책)")

In [ ]:
# 셀 5: [graph-mail 키] GET /users/{id}/messages → 200 기대
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_MAIL, params={"$top": 3})
assert r.status_code == 200, f"기대 200, 실제 {r.status_code}: {r.text[:300]}"
msgs = r.json().get("value", [])
print(f"메시지 {len(msgs)}건 조회 성공")

In [ ]:
# 셀 6: [graph-mail 키] GET /users → 403 기대 (정책 차단)
r = graph_get("/users", KEY_MAIL)
assert r.status_code == 403, f"기대 403(정책 차단), 실제 {r.status_code}: {r.text[:300]}"
print("✅ 예상대로 차단됨 — graph-mail 구독은 사용자 목록 조회 불가 (APIM 정책)")

## Part 2 (옵션 B): 권한별 UAMI

`scripts/deploy-graph-uami.sh` 실행 + 정책을 Part 2 버전(`client-id` 라우팅)으로 교체한 뒤 실행하세요.

이제 격리는 **정책이 아니라 토큰 권한 자체**로 이뤄집니다.
`graph-users` 경로가 잘못 열려도, 그 UAMI 토큰에는 `Mail.Read`가 없어 **Graph가 직접 거부**합니다.

In [ ]:
# 셀 7: [graph-users 키] 메일 접근 재시도 → Graph 권한 거부 기대 (403)
# Part 1과 달리, 이 403은 APIM 정책이 아니라 Graph가 "권한 없음"으로 반환합니다.
# (정책 게이트를 제거/완화한 Part 2 정책에서 테스트)
r = graph_get(f"/users/{TEST_USER_ID}/messages", KEY_USERS)
print("본문 일부:", r.text[:300])
assert r.status_code in (401, 403), f"기대 401/403(Graph 권한 거부), 실제 {r.status_code}"
print("✅ UAMI 토큰에 Mail.Read가 없어 Graph가 거부 — 진짜 격리 확인")

## Part 1 vs Part 2 요약

| 구분 | 옵션 A (System MI + 정책) | 옵션 B (권한별 UAMI) |
|------|--------------------------|----------------------|
| 실제 권한 경계 | MI 1개 = 모든 권한 합집합 | UAMI별 단일 권한 |
| 차단 주체 | APIM 정책 (403) | Graph 자체 (토큰에 권한 없음) |
| 정책 우회 시 | 전체 권한 노출 위험 | 여전히 안전 (토큰에 권한 없음) |
| 구성 복잡도 | 낮음 (MI 1개) | 높음 (UAMI 2개 + attach) |

**공통점:** 클라이언트는 두 경우 모두 구독 키만 보내며, Graph 토큰을 직접 발급하지 않습니다.
**결론:** 최소 권한/강한 격리가 필요하면 옵션 B, 간단한 데모/내부용이면 옵션 A.